In [13]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

load = PyPDFLoader("../data/Sample Blood Test Report.pdf")
doc_load = load.load()
#print("doc_load",doc_load)
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
split_chunks = splitter.split_documents(doc_load)
#print(split_chunks)
embed = OpenAIEmbeddings(model = "text-embedding-3-large")
vector_store = InMemoryVectorStore.from_documents(
    documents=split_chunks,
    embedding= embed
)
llm = ChatGroq(model = "openai/gpt-oss-20b")
########Lang graph -> Defininig the State, Nodes and edge##########
class RagNode(BaseModel):
    question:str = Field(description="User Question Here..")
    document:list = []
    context: str = Field(default="")
    answer:str = Field(default="")

def retreive_document(state:RagNode) -> RagNode:
    get_doc = vector_store.similarity_search(query=state.question)
    state.document = get_doc
    return state

def get_context(state:RagNode) -> RagNode:
    context = ""

    for doc in state.document:
        context+=doc.page_content +"\n\n"
    state.context = context
    return state

def get_answer(state:RagNode) -> RagNode:
    prompt= f"""
        You are a assistant and provide the answer for user question based on the provided context.
        If you don't find the relevant answer, then just say 'I dont know.'.
        Context is: {state.context},
        Questions is: {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state

graph = StateGraph(RagNode)
graph.add_node("retrive_document",retreive_document)
graph.add_node("get_context",get_context)
graph.add_node("get_answer",get_answer)

graph.add_edge(START, "retrive_document")
graph.add_edge("retrive_document", "get_context")
graph.add_edge("get_context", "get_answer")
graph.add_edge("get_answer", END)

graph = graph.compile()

res = graph.invoke({"question":"How old is the patient"})
print(res["answer"])



The patient is 42 years old.
